In [1]:
# configを読み取る
import datetime
import os
import yaml
import json

config_path = "/workspace/configs/base_config.yaml"

# read config
config = yaml.safe_load(open(config_path, "r"))

# get parameters from config
video_dir = config["video_dir"]
output_dir_top = config["output_dir"]
groundingdino_config_path = config["groundingdino"]["config_path"]
groundingdino_checkpoint = config["groundingdino"]["checkpoint"]

# read annotation
annotation_path = config["annotation_path"]
annotation = [json.loads(line) for line in open(annotation_path, "r").readlines()]

videos_paths = [f"{video_dir}/{a['video_path']}" for a in annotation]
instructions = [a["instruction"] for a in annotation]

# 出力フォルダを作成
# time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
time_stamp = "test"
output_dir = f"{output_dir_top}/{time_stamp}"
os.makedirs(output_dir, exist_ok=True)

# 本番では使わない
subclasses = [a["selected_subclass"] for a in annotation]

In [2]:
# 'selected_subclass': 'Zoom in', の array id 探す
target_subclass = "Dolly in"
target_ids = [i for i, a in enumerate(subclasses) if a == target_subclass]
local_target_id = 0
target_instruction = instructions[target_ids[local_target_id]]
target_video_path  = videos_paths[target_ids[local_target_id]]

print(f"num of target_subclass: {len(target_ids)}")
print(f"target_instruction: {target_instruction}")
print(f"target_video_path: {target_video_path}")

num of target_subclass: 3
target_instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
target_video_path: /workspace/data/videos/wyzi9GNZFMU_0_0to121.mp4


In [3]:
# instruction parser を読み込む
# /workspace/src にパスを通す
import sys
sys.path.append("/workspace/src")
from parse.instruction_parser_v3_rulebase_trial013_singlefile_kai2 import (
    build_parser
)
parser = build_parser()
request = parser.infer(target_instruction)
action = request["action"]
target = request["target"]
print(f"request: {request}")

request: {'action': 'dolly_in', 'target': "man's face", 'constraints': [], 'params': {}}


## parserの既存関数の確認

In [4]:
# load video
from utils.video_utility import load_video, write_video, show_video
frames, fps, width, height = load_video(target_video_path)

print(f"num of frames: {len(frames)}")
print(f"fps: {fps}, width: {width}, height: {height}")

# mp4を表示
show_video(target_video_path)

num of frames: 120
fps: 23.976, width: 1920, height: 1080


In [5]:
from postprocess.dispatcher import run_method
import logging

# {'action': 'zoom_in', 'target': 'camera_view', 'constraints': [], 'params': {}}
out_frames = run_method(
    action = request["action"],
    targets = "person", #[request["target"]],
    frames = frames[:10],
    params = request.get("params", {}),
    instruction = target_instruction,
    logger = logging.getLogger("zoom_in")
)

out_path = f"{output_dir}/zoom_in_output_2.mp4"
write_video(out_path, out_frames, fps, width, height)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------- dolly_in ---------------


dolly_in:object_zoom_in [unknown.mp4]:   0%|          | 0/10 [00:00<?, ?frame/s]FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers


final text_encoder_type: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15473.98it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# 結果の確認
show_video(out_path)
frames, fps, width, height = load_video(out_path)
print(f"num of frames: {len(frames)}")
print(f"fps: {fps}, width: {width}, height: {height}")

num of frames: 10
fps: 23.976, width: 1920, height: 1080
